In [102]:
import uuid

In [98]:


session_id = str(uuid.uuid4())

print("Session ID:", session_id)

Session ID: dca84c94-f4c2-47b5-857f-4f26db0988d4


In [7]:
import openai
import dotenv

print("Packages working!")


Packages working!


In [8]:
from dotenv import load_dotenv
import os

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

print("Endpoint:", endpoint)
print("Deployment:", deployment)

Endpoint: https://employee-assist-foundryy.services.ai.azure.com
Deployment: None


In [9]:
from dotenv import load_dotenv
import os

load_dotenv()

endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
key = os.getenv("AZURE_SEARCH_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX")

print("Endpoint:", endpoint)
print("Index:", index_name)
print("Key loaded:", key is not None)

Endpoint: https://employee-assist-search.search.windows.net
Index: employee-assist-index
Key loaded: True


In [10]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

search_client = SearchClient(
    endpoint=endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(key)
)

print("Connected to Azure AI Search!")

Connected to Azure AI Search!


In [117]:
from azure.storage.blob import BlobServiceClient
import os

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
container_name = os.getenv("AZURE_STORAGE_CONTAINER")

blob_service_client = BlobServiceClient.from_connection_string(
    connection_string
)

container_client = blob_service_client.get_container_client(container_name)

for blob in container_client.list_blobs():
    print(blob.name)

Base_Locations.txt
Genome Learning Policy.pdf
Genpact.txt
India Medical Insurance _ FAQs 2026-27.pdf
Leave Policy.pdf
Microsoft Word - GENPACT INDIA BACKGROUND CHECK PROCEDURES.pdf
Overtime Policy  Procedure.pdf
Presentation PowerPoint.pdf
Telecom Reimbursement Policy.pdf
Timesheet Policy.pdf


In [12]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("Storage connection loaded:", os.getenv("AZURE_STORAGE_CONNECTION_STRING") is not None)
print("Container:", os.getenv("AZURE_STORAGE_CONTAINER"))

Storage connection loaded: True
Container: documents


Extracting text from Files

In [119]:
from pypdf import PdfReader
from io import BytesIO

# Processing PDFs and TXT files from Blob Storage
documents = []

for blob in container_client.list_blobs():

    blob_name = blob.name

    print(f"Processing: {blob_name}")

    # Get the blob
    blob_client = container_client.get_blob_client(blob_name)

    # Download file into memory
    file_data = blob_client.download_blob().readall()


    if blob_name.lower().endswith(".pdf"):

        reader = PdfReader(BytesIO(file_data))

        for page_number, page in enumerate(reader.pages, start=1):

            page_text = page.extract_text()

            if page_text and page_text.strip():

                documents.append({
                    "source": blob_name,
                    "page": page_number,
                    "content": page_text.strip()
                })


    elif blob_name.lower().endswith(".txt"):

        text = file_data.decode("utf-8", errors="ignore").strip()

        if text:

            documents.append({
                "source": blob_name,
                "page": 1,
                "content": text
            })



    else:
        print(f"Skipping unsupported file: {blob_name}")


print("\n-------------------------------")
print("EXTRACTION COMPLETE")
print("---------------------------------")

print("Total sections extracted:", len(documents))

print("\nFiles processed:")

for file in sorted(set(doc["source"] for doc in documents)):
    print("-", file)

Processing: Base_Locations.txt
Processing: Genome Learning Policy.pdf
Processing: Genpact.txt
Processing: India Medical Insurance _ FAQs 2026-27.pdf
Processing: Leave Policy.pdf
Processing: Microsoft Word - GENPACT INDIA BACKGROUND CHECK PROCEDURES.pdf
Processing: Overtime Policy  Procedure.pdf
Processing: Presentation PowerPoint.pdf
Processing: Telecom Reimbursement Policy.pdf
Processing: Timesheet Policy.pdf

-------------------------------
EXTRACTION COMPLETE
---------------------------------
Total sections extracted: 86

Files processed:
- Base_Locations.txt
- Genome Learning Policy.pdf
- Genpact.txt
- India Medical Insurance _ FAQs 2026-27.pdf
- Leave Policy.pdf
- Microsoft Word - GENPACT INDIA BACKGROUND CHECK PROCEDURES.pdf
- Overtime Policy  Procedure.pdf
- Presentation PowerPoint.pdf
- Telecom Reimbursement Policy.pdf
- Timesheet Policy.pdf


Chunking

In [120]:
# Chunking configuration
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = []

for doc in documents:
    text = doc["content"]
    source = doc["source"]
    page = doc["page"]

    start = 0

    while start < len(text):
        end = start + CHUNK_SIZE

        chunk_text = text[start:end].strip()

        if chunk_text:
            chunks.append({
                "source": source,
                "page": page,
                "content": chunk_text
            })

        start += CHUNK_SIZE - CHUNK_OVERLAP

print("Total chunks:", len(chunks))

Total chunks: 257


Checking Chunks

In [123]:
print(chunks[0])
print(chunks[1])

for i, chunk in enumerate(chunks[:5]):
    print(f"\n--- Chunk {i+1} ---")
    print("Source:", chunk["source"])
    print("Page:", chunk["page"])
    print("Content:", chunk["content"][:300])

{'source': 'Base_Locations.txt', 'page': 1, 'content': 'Genpact Ltd. Headquarters and Office Locations\r\nGenpact Ltd. is a leading American information technology services, consulting, and outsourcing company founded in 1997 by Pramod Bhasin. Genpact Ltd. is headquartered in New York City and operates primarily in the IT, consulting, and outsourcing sectors. With a global workforce of approximately 125,000 employees, Genpact Ltd. specializes in business process outsourcing and digital transformation services for clients around the world.\r\n\r\nGenpact Ltd. has established a significant corporate office presence across the globe. The Genpact Ltd. corporate office network is anchored by its main headquarters in New York, USA, which oversees global operations and houses key leadership functions. In addition to its New York headquarters, Genpact Ltd. maintains a separate legal headquarters in Bermuda and has major offices, technology hubs, and delivery centers spanning North America, Lat

Generating embeddings for all the chunks


In [124]:


texts = [chunk["content"] for chunk in chunks]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

for i, item in enumerate(response.data):
    chunks[i]["contentVector"] = item.embedding

print("Embeddings generated:", len(chunks))
print("Vector dimensions:", len(chunks[0]["contentVector"]))

Embeddings generated: 257
Vector dimensions: 1536


Preparing documents for Azure AI Search

In [125]:


search_documents = []

for i, chunk in enumerate(chunks):

    search_documents.append({
        "id": f"doc-{i}",
        "content": chunk["content"],
        "source": chunk["source"],
        "page": chunk["page"],
        "contentVector": chunk["contentVector"]
    })

print("Documents ready:", len(search_documents))

Documents ready: 257


 Uploading docs to Azure AI Search

In [126]:


result = search_client.upload_documents(
    documents=search_documents
)

print("Upload completed!")
print(result)


Upload completed!
[{'key': 'doc-0', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-1', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-2', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-3', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-4', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-5', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-6', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-7', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-8', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-9', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-10', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-11', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'key': 'doc-12', 'status': True, 'errorMessage': None, 'statusCode': 200}, {'k

In [18]:
from openai import AzureOpenAI
import os

load_dotenv(override=True)

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-10-21"
)

response = client.embeddings.create(
    model="text-embedding-3-small",
    input="This is a test document about employee leave policy."
)

embedding = response.data[0].embedding

print("Embedding generated!")
print("Dimensions:", len(embedding))
print("First 5 values:", embedding[:5])

Embedding generated!
Dimensions: 1536
First 5 values: [0.0017833709716796875, 0.050872802734375, 0.05548095703125, -0.0038127899169921875, -0.01331329345703125]


Uploading vectors to Azure AI search (SSL setup)


In [1]:
import truststore

truststore.inject_into_ssl()

print("SSL setup complete")

SSL setup complete


In [3]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

endpoint = os.getenv("AZURE_SEARCH_ENDPOINT")
key = os.getenv("AZURE_SEARCH_KEY")
index_name = os.getenv("AZURE_SEARCH_INDEX")

print("Endpoint:", endpoint)
print("Index:", index_name)
print("Key loaded:", key is not None)

Endpoint: https://employee-assist-search.search.windows.net
Index: employee-assist-index
Key loaded: True


In [4]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

search_client = SearchClient(
    endpoint=endpoint,
    index_name=index_name,
    credential=AzureKeyCredential(key)
)

print("Search client recreated")

Search client recreated


In [22]:
result = search_client.upload_documents(
    documents=search_documents[:1]
)

print(result)

[{'key': 'doc-0', 'status': True, 'errorMessage': None, 'statusCode': 200}]


In [129]:
query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=question
)

query_vector = query_response.data[0].embedding

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="contentVector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["content", "source", "page"],
    top=5
)

results = list(results)

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Content:", result["content"][:500])


--- Result 1 ---
Source: Leave Policy.pdf
Page: 2
Content: Leave Policy 
         
 
 
  © 2025 Copyright Genpact. All Rights Reserved  
PURPOSE 
The objective of this policy is to state the entitlement, procedure and guidelines for the leave that can be availed of by 
the Employees (as defined in this policy) 
 
DEFINITION 
Company / Genpact: Genpact India Private Limited and will include any/ all of its affiliate/s and/or group company/ies 
 registered in India. Affiliates and/or group company/ies would mean and include companies having common ultimat

--- Result 2 ---
Source: Leave Policy.pdf
Page: 3
Content: Leave Policy 
         
 
 
  © 2025 Copyright Genpact. All Rights Reserved  
ELIGIBILITY 
All Full time, Part time and Fixed Term Employees of Genpact 
 
EXCLUSIONS 
1. Some Group Companies may have a different policy and the same shall be applicable to their employees.  
2. Contractors, Vendors, Temp resources, Apprentices/Interns 
 
 
GUIDELINES 
A. EARNED LEAVE 
Entitlemen

Testing retrieval

In [128]:
results = search_client.search(
    search_text="What is the purpose of the Overtime policy?"
)

for result in results:
    print("\n--- Result ---")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Content:", result["content"][:500])


--- Result ---
Source: Overtime Policy  Procedure.pdf
Page: 2
Content: d 
by a designated person as prescribed in this policy or procedures that is communicated from 
time to time. 
 
c. Eligible Employee: All full-time employees who are designated as Band 5 employees working 
in Genpact in India will be eligible. 
 
d. Overtime: Authorized Hours worked by a n Eligible Employee in excess of  Statutory working 
hours and after obtaining approvals in accordance with this policy or procedures as 
communicated from time to time . Overtime period shall not exceed the li

--- Result ---
Source: Overtime Policy  Procedure.pdf
Page: 4
Content: mplete their work within the  Statutory working 
hours; 
c. Overtime, when necessary shall be initiated by the supervisor or business only; 
d. Overtime shall at all times be at the discretion of the Company; 
e. Compensatory time off is not allowed in lieu of pay for Overtime worked. 
f. Eligible Employees will not be entitled to Overtime for:  
i. tra

In [131]:
print(question)

What is the purpose of the leave policy?


Sending query vector to Azure AI search

In [ ]:


from azure.search.documents.models import VectorizedQuery

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="contentVector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["content", "source", "page"],
    top=5
)

for i, result in enumerate(results, start=1):
    print(f"\n----------- Result {i} -------------")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Content:", result["content"][:500])


----------- Result 1 -------------
Source: Leave Policy.pdf
Page: 2
Content: Leave Policy 
         
 
 
  © 2025 Copyright Genpact. All Rights Reserved  
PURPOSE 
The objective of this policy is to state the entitlement, procedure and guidelines for the leave that can be availed of by 
the Employees (as defined in this policy) 
 
DEFINITION 
Company / Genpact: Genpact India Private Limited and will include any/ all of its affiliate/s and/or group company/ies 
 registered in India. Affiliates and/or group company/ies would mean and include companies having common ultimat

----------- Result 2 -------------
Source: Leave Policy.pdf
Page: 3
Content: Leave Policy 
         
 
 
  © 2025 Copyright Genpact. All Rights Reserved  
ELIGIBILITY 
All Full time, Part time and Fixed Term Employees of Genpact 
 
EXCLUSIONS 
1. Some Group Companies may have a different policy and the same shall be applicable to their employees.  
2. Contractors, Vendors, Temp resources, Apprentices/Interns 
 
 
GUI

In [132]:
from openai import AzureOpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)

chat_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-10-21"
)

chat_deployment = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT")

print("Chat client connected!")
print("Deployment:", chat_deployment)

Chat client connected!
Deployment: gpt-5-mini


LLM Generation Setup

In [133]:
response = chat_client.chat.completions.create(
    model=chat_deployment,
    messages=[
        {
            "role": "user",
            "content": "Explain what a leave policy is in one sentence."
        }
    ],
    max_completion_tokens=500
)

print(response.choices[0].message.content)

A leave policy is an organization's formal set of rules that defines employees' entitlements to time off (types, eligibility and duration), whether it is paid or unpaid, and the procedures for requesting, approving, and documenting that leave.


In [ ]:
from datetime import datetime, timezone
import uuid

chat_record = {
    "id": str(uuid.uuid4()),
    "sessionId": "demo-session-001",
    "question": "What types of leave are available to employees?",
    "answer": answer,
    "sources": sources,
    "timestamp": datetime.now(timezone.utc).isoformat()
}

container.create_item(body=chat_record)

print("Chat record stored in Cosmos DB!")

In [34]:
print(type(results))
print(len(list(results)))

<class 'azure.search.documents._operations._patch.SearchItemPaged'>
0


Hardcoding the query

In [139]:
question = input("Enter your question: ")

print("Question:", question)

Question: what is Genome


In [140]:
query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=question
)

query_vector = query_response.data[0].embedding

vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="contentVector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["content", "source", "page"],
    top=5
)

results = list(results)

print("Retrieved results:", len(results))

Retrieved results: 5


Building the context from the retrieved chunks

In [141]:
context = ""

for i, result in enumerate(results, start=1):
    context += f"""
Source: {result['source']}
Page: {result['page']}
Content:
{result['content']}

---
"""

print(context[:3000])


Source: Genome Learning Policy.pdf
Page: 3
Content:
itions 
 
 
4.1 “Learner” means all regular full-time and part-time employees of the Company who 
have access to Genome for learning.

---

Source: Genome Learning Policy.pdf
Page: 4
Content:
4 of 6  
 
 
 Classification : Genpact Internal 
4.2  “Genome” is the learning system in Company and is an official system of records for 
the time a Learner has spent in learning.  
Genome has 4 steps (details in the link):  
• B – skill inventory 
• I – self-guided learning 
• T –  learning with others, and  
• S – learning by doing 
 
5. Role & Responsibility 
 
5.1 Reporting managers are responsible for: 
(a) providing support and guidance in relation to learning on Genome  
(b) identifying learning and development needs 
(c) facilitating access to learning opportunities in line with the Learner’s learning 
needs  
(d) addressing problematic performance (where appropriate), and 
(e) making sure that appropriate action is taken accordingly. 


Sending the retrieved context with question to GPT - Imp Step

In [142]:
prompt = f"""
You are an employee policy assistant.

Answer the user's question using ONLY the information provided in the context.

Rules:
- Use only the provided context.
- Do not use outside knowledge.
- If the context contains multiple points, present them as bullet points.
- If the answer is one simple fact, give one concise sentence.
- If the answer is not available in the context, say:
  "I couldn't find that information in the provided documents."

Context:
{context}

User question:
{question}
"""

response = chat_client.chat.completions.create(
    model=chat_deployment,
    messages=[
        {
            "role": "system",
            "content": "Answer questions only using the provided document context."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_completion_tokens=2000
)

answer = response.choices[0].message.content

print("ANSWER:")
print(answer)

ANSWER:
- Genome is the learning system in the Company and is an official system of records for the time a Learner has spent in learning.  
- It is a learning and reskilling platform.  
- Genome has four steps: B – skill inventory; I – self-guided learning; T – learning with others; S – learning by doing.


Connect the answer to Cosmos DB

In [143]:
import uuid
from datetime import datetime, timezone

chat_record = {
    "id": str(uuid.uuid4()),
    "sessionId": "demo-session-001",
    "question": question,
    "answer": answer,
    "sources": [
        {
            "source": result["source"],
            "page": result["page"]
        }
        for result in results
    ],
    "timestamp": datetime.now(timezone.utc).isoformat()
}

container.create_item(body=chat_record)

print("Saved to Cosmos DB!")

Saved to Cosmos DB!


In [144]:
history = get_chat_history("demo-session-001")

print("Chat history:")

for chat in history:
    print("\nQuestion:", chat["question"])
    print("Answer:", chat["answer"])
    print("Time:", chat["timestamp"])

Chat history:

Question: What types of leave are available to employees?
Answer: I found the following leave types in the provided policy:

- Earned Leave (paid, accrued based on days worked)  
- Casual Leave (paid, for unforeseen events)  
- Sick Leave (paid, for illness)  
- Maternity Leave (paid, for women for childbirth, commissioning mothers, adoption, related illness/miscarriage)  
- Paternity Leave (paid; generally 5 working days, 84 calendar days for a single male parent)  
- Compensatory Off (paid holiday in lieu of working on a national/festival holiday or weekly off)  
- National & Festival Holidays (entitlement of paid holiday days, separate from annual leave)  
- Leave Without Pay (LWOP) (unpaid leave option referenced in the policy)
Time: 2026-09-17T09:36:17.166817+00:00

Question: How many leaves are available to employees in one year?
Answer: Earned leave: 18 paid days per year (employees in Rajasthan: 30).  
Casual leave: 6 days per year (except Rajasthan).  
Sick leav

In [145]:
import uuid

session_id = str(uuid.uuid4())

print("Session ID:", session_id)

Session ID: 65ce174d-cd30-4471-ab07-67dcb6efc653


Cosmos DB history available to the RAG prompt

In [146]:
history = get_chat_history(session_id)

print("Previous messages in this session:", len(history))

for chat in history:
    print("\nQuestion:", chat["question"])
    print("Answer:", chat["answer"])

Previous messages in this session: 0


In [147]:
question = "How many earned leaves are employees entitled to in a year?"

print("Question:", question)

Question: How many earned leaves are employees entitled to in a year?


In [148]:
query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=question
)

query_vector = query_response.data[0].embedding

print("Query embedding generated!")
print("Dimensions:", len(query_vector))

Query embedding generated!
Dimensions: 1536


In [149]:
vector_query = VectorizedQuery(
    vector=query_vector,
    k_nearest_neighbors=5,
    fields="contentVector"
)

results = search_client.search(
    search_text=None,
    vector_queries=[vector_query],
    select=["content", "source", "page"],
    top=5
)

results = list(results)

print("Retrieved results:", len(results))

for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Source:", result["source"])
    print("Page:", result["page"])
    print("Content:", result["content"][:300])

Retrieved results: 5

--- Result 1 ---
Source: Leave Policy.pdf
Page: 3
Content: liday) to earned leave will not be counted as part 
of earned leave 
• In case an employee avails leaves more than the existing leave balance, these additional leaves will be unpaid. 
 However, with approval of HR Manager / Supervisor, a maximum of 5 days of paid leave may be allowed over and above 

--- Result 2 ---
Source: Leave Policy.pdf
Page: 3
Content: Leave Policy 
         
 
 
  © 2025 Copyright Genpact. All Rights Reserved  
ELIGIBILITY 
All Full time, Part time and Fixed Term Employees of Genpact 
 
EXCLUSIONS 
1. Some Group Companies may have a different policy and the same shall be applicable to their employees.  
2. Contractors, Vendors, T

--- Result 3 ---
Source: Leave Policy.pdf
Page: 10
Content: If an employee in Telangana has an unutilized leave balance of 70 days at the end of the year, then 60 leaves 
will be carried over to next year and 10 days will be encashed and paid in January pa

In [150]:
context = ""

for i, result in enumerate(results, start=1):
    context += f"""
[CHUNK {i}]
Source: {result["source"]}
Page: {result["page"]}
Content:
{result["content"]}

---
"""

print(context[:3000])


[CHUNK 1]
Source: Leave Policy.pdf
Page: 3
Content:
liday) to earned leave will not be counted as part 
of earned leave 
• In case an employee avails leaves more than the existing leave balance, these additional leaves will be unpaid. 
 However, with approval of HR Manager / Supervisor, a maximum of 5 days of paid leave may be allowed over and above the 
leave balance which will be adjusted from the future leave accruals 
Carry Forward 
• Earned leave balance at the end of the year can be carried forward to the following year up to a maximum of 30 days 
(applicable for all locations except (a) Telangana, where the maximum carry forward is 60 days, (b) Gujarat, where the maximum 
carry forward is 63 days and (c) Karnataka, Tamil Nadu and Maharashtra where the maximum carry forward is 45 days) 
Encashment 
Treatment of any earned leave balance in excess of the annual carry forward limit mentioned above, if not availed of during the 
year, shall be as follows:  
• For Band 4 and above em

In [151]:
history = get_chat_history(session_id)

conversation_history = ""

for chat in history:
    conversation_history += f"""
Previous User Question:
{chat["question"]}

Previous Assistant Answer:
{chat["answer"]}

---
"""

print(conversation_history)

In [152]:
prompt = f"""
You are an employee policy assistant.

Answer the user's question using ONLY the information provided in the context.

Rules:
- Use only the provided context.
- Do not use outside knowledge.
- If the answer contains multiple points, present them as bullet points.
- If the answer is one simple fact, give one concise sentence.
- If the answer is not available in the context, say:
  "I couldn't find that information in the provided documents."

Context:
{context}

User question:
{question}
"""

response = chat_client.chat.completions.create(
    model=chat_deployment,
    messages=[
        {
            "role": "system",
            "content": "Answer questions only using the provided document context."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_completion_tokens=2000
)

answer = response.choices[0].message.content

print("ANSWER:")
print(answer)

ANSWER:
All employees are entitled to 18 paid leaves in a year, except employees working in Rajasthan who are entitled to 30 days.


In [153]:
chat_record = {
    "id": str(uuid.uuid4()),
    "sessionId": session_id,
    "question": question,
    "answer": answer,
    "sources": [
        {
            "source": result["source"],
            "page": result["page"]
        }
        for result in results
    ],
    "timestamp": datetime.now(timezone.utc).isoformat()
}

container.create_item(body=chat_record)

print("Saved to Cosmos DB!")

Saved to Cosmos DB!


In [154]:
question = "What about employees in Rajasthan?"

print("Question:", question)

Question: What about employees in Rajasthan?


In [155]:
history = get_chat_history(session_id)

conversation_history = ""

for chat in history:
    conversation_history += f"""
Previous User Question:
{chat["question"]}

Previous Assistant Answer:
{chat["answer"]}

---
"""

print(conversation_history)


Previous User Question:
How many earned leaves are employees entitled to in a year?

Previous Assistant Answer:
All employees are entitled to 18 paid leaves in a year, except employees working in Rajasthan who are entitled to 30 days.

---



In [156]:
prompt = f"""
You are an employee policy assistant.

Answer the current question using ONLY:
1. The provided conversation history
2. The provided document context

Rules:
- Use the conversation history to understand references such as "what about them", "there", or "that".
- Use the document context as the factual source.
- Do not use outside knowledge.
- If the answer contains multiple points, present them as bullet points.
- If the answer is one simple fact, give one concise sentence.
- If the answer is not available in the context, say:
  "I couldn't find that information in the provided documents."

Conversation history:
{conversation_history}

Document context:
{context}

Current user question:
{question}
"""

response = chat_client.chat.completions.create(
    model=chat_deployment,
    messages=[
        {
            "role": "system",
            "content": "Answer questions only using the provided conversation history and document context."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    max_completion_tokens=2000
)

answer = response.choices[0].message.content

print("ANSWER:")
print(answer)

ANSWER:
Employees working in Rajasthan are entitled to 30 paid earned leaves in a year.


In [157]:
chat_record = {
    "id": str(uuid.uuid4()),
    "sessionId": session_id,
    "question": question,
    "answer": answer,
    "sources": [
        {
            "source": result["source"],
            "page": result["page"]
        }
        for result in results
    ],
    "timestamp": datetime.now(timezone.utc).isoformat()
}

container.create_item(body=chat_record)

print("Follow-up conversation saved to Cosmos DB!")

Follow-up conversation saved to Cosmos DB!


ask_rag() function

In [158]:
import uuid
import json
from datetime import datetime, timezone

In [159]:
def ask_rag(question, session_id=None):

    # 1. Create a session if one isn't provided
    if session_id is None:
        session_id = str(uuid.uuid4())

    # 2. Get previous conversation from Cosmos DB
    history = get_chat_history(session_id)

    conversation_history = ""

    for chat in history:
        conversation_history += f"""
Previous User Question:
{chat["question"]}

Previous Assistant Answer:
{chat["answer"]}

---
"""

    # 3. Create embedding for the current question
    query_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    )

    query_vector = query_response.data[0].embedding

    # 4. Search Azure AI Search
    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=5,
        fields="contentVector"
    )

    results = search_client.search(
        search_text=None,
        vector_queries=[vector_query],
        select=["content", "source", "page"],
        top=5
    )

    results = list(results)

    # 5. Build document context
    context = ""

    for i, result in enumerate(results):
        context += f"""
[CHUNK {i}]
Source: {result["source"]}
Page: {result["page"]}
Content:
{result["content"]}

---
"""

    # 6. Create RAG prompt
    prompt = f"""
You are an employee policy assistant.

Answer the current question using ONLY:
1. The provided conversation history
2. The provided document context

Rules:
- Use conversation history to understand follow-up questions.
- Use the document context as the factual source.
- Do not use outside knowledge.
- If the answer contains multiple points, present them as bullet points.
- If the answer is one simple fact, give one concise sentence.
- Do not invent or assume information.
- Identify the single CHUNK that most directly supports the answer.
- If the answer is not available in the context, use:
  "I couldn't find that information in the provided documents."

Return ONLY valid JSON.

If the answer is found:
{{
    "answer": "your answer here",
    "source_chunk": 0
}}

If the answer is not found:
{{
    "answer": "I couldn't find that information in the provided documents.",
    "source_chunk": null
}}

Conversation history:
{conversation_history}

Document context:
{context}

Current user question:
{question}
"""

    # 7. Send prompt to Azure OpenAI
    response = chat_client.chat.completions.create(
        model=chat_deployment,
        messages=[
            {
                "role": "system",
                "content": "Answer questions using only the provided conversation history and document context."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        max_completion_tokens=2000
    )

    response_text = response.choices[0].message.content

    # 8. Parse the JSON response
    try:
        result_data = json.loads(response_text)

        answer = result_data.get("answer", "")
        source_chunk = result_data.get("source_chunk")

    except json.JSONDecodeError:
        answer = response_text
        source_chunk = None

    # 9. Get the single supporting source
    sources = []

    if source_chunk is not None:

        try:
            source_chunk = int(source_chunk)

            if 0 <= source_chunk < len(results):

                result = results[source_chunk]

                sources.append({
                    "source": result["source"],
                    "page": result["page"]
                })

        except (ValueError, TypeError):
            pass

    # 10. Save conversation to Cosmos DB
    chat_record = {
        "id": str(uuid.uuid4()),
        "sessionId": session_id,
        "question": question,
        "answer": answer,
        "sources": sources,
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

    container.create_item(body=chat_record)

    # 11. Return answer and source
    return answer, sources

In [163]:
print("Sources returned by ask_rag():")
print(sources)

Sources returned by ask_rag():
[{'source': 'Leave Policy.pdf', 'page': 3}]


In [160]:
answer, sources = ask_rag(
    "How many earned leaves are employees entitled to in a year?",
    session_id
)
print("ANSWER:")
print(answer)

print("\nSOURCES:")

if sources:
    for source in sources:
        print(f"- {source['source']} — Page {source['page']}")
else:
    print("No source found.")

ANSWER:
- All employees are entitled to 18 paid leaves in a year.
- Employees working in Rajasthan are entitled to 30 paid leaves in a year.

SOURCES:
- Leave Policy.pdf — Page 3


In [161]:
answer, sources = ask_rag(
    "What about employees in Rajasthan?",
    session_id
)

print("ANSWER:")
print(answer)

print("\nSOURCES:")
for source in sources:
    print(f"- {source['source']} — Page {source['page']}")

ANSWER:
Employees working in Rajasthan are entitled to 30 paid earned leaves in a year.

SOURCES:
- Leave Policy.pdf — Page 3


Verifying Cosmos DB conversation history

In [162]:
history = get_chat_history(session_id)

print("Messages in this session:", len(history))

for i, chat in enumerate(history, start=1):
    print(f"\n--- Message {i} ---")
    print("Question:", chat["question"])
    print("Answer:", chat["answer"])
    print("Sources:", chat.get("sources"))

Messages in this session: 4

--- Message 1 ---
Question: How many earned leaves are employees entitled to in a year?
Answer: All employees are entitled to 18 paid leaves in a year, except employees working in Rajasthan who are entitled to 30 days.
Sources: None

--- Message 2 ---
Question: What about employees in Rajasthan?
Answer: Employees working in Rajasthan are entitled to 30 paid earned leaves in a year.
Sources: None

--- Message 3 ---
Question: How many earned leaves are employees entitled to in a year?
Answer: - All employees are entitled to 18 paid leaves in a year.
- Employees working in Rajasthan are entitled to 30 paid leaves in a year.
Sources: None

--- Message 4 ---
Question: What about employees in Rajasthan?
Answer: Employees working in Rajasthan are entitled to 30 paid earned leaves in a year.
Sources: None


In [167]:
container.create_item(body=chat_record)

print("Record saved to Cosmos DB with source!")

Record saved to Cosmos DB with source!


In [172]:
def get_chat_history(session_id):

    query = """
    SELECT c.question, c.answer, c.sources, c.timestamp
    FROM c
    WHERE c.sessionId = @sessionId
    ORDER BY c.timestamp ASC
    """

    parameters = [
        {
            "name": "@sessionId",
            "value": session_id
        }
    ]

    items = container.query_items(
        query=query,
        parameters=parameters,
        enable_cross_partition_query=False
    )

    history = list(items)

    return history

In [173]:
history = get_chat_history(session_id)

for i, chat in enumerate(history, start=1):
    print(f"\n--- Message {i} ---")
    print("Question:", chat["question"])
    print("Answer:", chat["answer"])
    print("Sources:", chat.get("sources"))


--- Message 1 ---
Question: How many earned leaves are employees entitled to in a year?
Answer: All employees are entitled to 18 paid leaves in a year, except employees working in Rajasthan who are entitled to 30 days.
Sources: [{'source': 'Leave Policy.pdf', 'page': 3}, {'source': 'Leave Policy.pdf', 'page': 3}, {'source': 'Leave Policy.pdf', 'page': 10}, {'source': 'Leave Policy.pdf', 'page': 10}, {'source': 'Leave Policy.pdf', 'page': 9}]

--- Message 2 ---
Question: What about employees in Rajasthan?
Answer: Employees working in Rajasthan are entitled to 30 paid earned leaves in a year.
Sources: [{'source': 'Leave Policy.pdf', 'page': 3}, {'source': 'Leave Policy.pdf', 'page': 3}, {'source': 'Leave Policy.pdf', 'page': 10}, {'source': 'Leave Policy.pdf', 'page': 10}, {'source': 'Leave Policy.pdf', 'page': 9}]

--- Message 3 ---
Question: How many earned leaves are employees entitled to in a year?
Answer: - All employees are entitled to 18 paid leaves in a year.
- Employees working

In [166]:
chat_record = {
    "id": str(uuid.uuid4()),
    "sessionId": session_id,
    "question": question,
    "answer": answer,
    "sources": sources,
    "timestamp": datetime.now(timezone.utc).isoformat()
}

print(chat_record)

{'id': 'ed0d158f-9008-4fd6-bfdd-d53404933eb7', 'sessionId': '65ce174d-cd30-4471-ab07-67dcb6efc653', 'question': 'What about employees in Rajasthan?', 'answer': 'Employees working in Rajasthan are entitled to 30 paid earned leaves in a year.', 'sources': [{'source': 'Leave Policy.pdf', 'page': 3}], 'timestamp': '2026-09-17T16:20:33.271120+00:00'}


In [45]:
from azure.cosmos import CosmosClient
import os

load_dotenv(override=True)

cosmos_client = CosmosClient(
    os.getenv("AZURE_COSMOS_ENDPOINT"),
    os.getenv("AZURE_COSMOS_KEY")
)

database = cosmos_client.get_database_client(
    os.getenv("AZURE_COSMOS_DATABASE")
)

container = database.get_container_client(
    os.getenv("AZURE_COSMOS_CONTAINER")
)

print("Connected to Cosmos DB!")

Connected to Cosmos DB!


In [82]:
def get_chat_history(session_id):

    query = """
    SELECT c.question, c.answer, c.timestamp
    FROM c
    WHERE c.sessionId = @sessionId
    ORDER BY c.timestamp ASC
    """

    parameters = [
        {
            "name": "@sessionId",
            "value": session_id
        }
    ]

    items = container.query_items(
        query=query,
        parameters=parameters,
        enable_cross_partition_query=False
    )

    history = list(items)

    return history

In [83]:
history = get_chat_history(session_id)

print("Chat history:")

for chat in history:
    print("\nQuestion:", chat["question"])
    print("Answer:", chat["answer"])

Chat history:

Question: How many earned leaves are employees entitled to in a year?
Answer: - All employees are entitled to 18 paid leaves in a year accrued month on month.
- Employees working in Rajasthan are entitled to 30 days in a year.


In [108]:
import os
from azure.cosmos import CosmosClient
from dotenv import load_dotenv

load_dotenv(override=True)

cosmos_client = CosmosClient(
    os.getenv("AZURE_COSMOS_ENDPOINT"),
    os.getenv("AZURE_COSMOS_KEY")
)

print("Cosmos client created")

database = cosmos_client.get_database_client(
    os.getenv("AZURE_COSMOS_DATABASE")
)

container = database.get_container_client(
    os.getenv("AZURE_COSMOS_CONTAINER")
)

print("Testing Cosmos connection...")

items = list(container.query_items(
    query="SELECT TOP 1 * FROM c",
    enable_cross_partition_query=True
))

print("Cosmos connection successful!")
print(items)

Cosmos client created
Testing Cosmos connection...
Cosmos connection successful!
[{'id': 'b1c49bd7-b015-47ea-b921-b5f57ed55552', 'sessionId': 'demo-session-001', 'question': 'What types of leave are available to employees?', 'answer': 'I found the following leave types in the provided policy:\n\n- Earned Leave (paid, accrued based on days worked)  \n- Casual Leave (paid, for unforeseen events)  \n- Sick Leave (paid, for illness)  \n- Maternity Leave (paid, for women for childbirth, commissioning mothers, adoption, related illness/miscarriage)  \n- Paternity Leave (paid; generally 5 working days, 84 calendar days for a single male parent)  \n- Compensatory Off (paid holiday in lieu of working on a national/festival holiday or weekly off)  \n- National & Festival Holidays (entitlement of paid holiday days, separate from annual leave)  \n- Leave Without Pay (LWOP) (unpaid leave option referenced in the policy)', 'sources': [{'source': 'Leave Policy.pdf', 'page': 2}, {'source': 'Leave Poli

In [174]:
answer, sources = ask_rag(
    "What about casual leave?",
    session_id
    )

print("ANSWER:")
print(answer)

print("\nSOURCES:")

if sources:
    for source in sources:
        print(f"- {source['source']} — Page {source['page']}")
else:
    print("No source found.")

ANSWER:
- All employees, except those working in Rajasthan, are entitled to 6 days of Casual leave in a year.
- For existing employees, 3 casual leaves are credited in January and the remaining 3 in July; for those who join mid‑year, leaves are proportionately credited.
- Intervening holidays to casual leave will not be counted as part of casual leave.
- Casual leaves are not carried over to the next year; any unavailed casual leave lapses on January 01 of the following year.
- Casual leave cannot be encashed.

SOURCES:
- Leave Policy.pdf — Page 4


In [179]:
answer, sources = ask_rag(
    "Who are eligible for Insurance?",
    session_id
)

print("ANSWER:")
print(answer)

print("\nSOURCES:")
if sources:
    for source in sources:
        print(f"- {source['source']} — Page {source['page']}")
else:
    print("No source found.")

ANSWER:
All Genpact India permanent employees are eligible for the insurance.

SOURCES:
- India Medical Insurance _ FAQs 2026-27.pdf — Page 3
